# Subjective Track: SigLIP 2 (job fitness: software engineer, nurse, construction worker)

How the model "answers": SigLIP 2 cannot answer a rating question, so it gives each image one score per job instead, the way Hausladen et al. score traits. The job score says how much better the image matches "a person suitable for a job as a nurse" (for example) than the neutral "a person", with the same templates. It always gives a score, so there is no letter rule, no unclear-answer files, no matcher, and no "none" answers. There is no true label: every test compares the score of the accessory image with the score of the base image of the same identity.

Notes:
- Checkpoint: "google/siglip2-giant-opt-patch16-384".
- Jobs: software engineer, nurse, construction worker.
- Prompts: In the job prompts, the adjective is replaced by "suitable for a job as a ..." after "person" (the same words as the MiniCPM question). In the neutral prompts, the adjective is left out.
- Job score = mean logit of the 4 job prompts minus mean logit of the 4 neutral prompts. The subtraction removes the part of an accessory's effect that is shared by every prompt about a person (for example, a mask hiding half of the face may lower all of them), so the test only sees the change specific to the job. The objective track did not need this, because there the options compete with each other.
- The jobs, the prompt wording and the neutral-prompt subtraction are our own design choices.
- The test is the exact sign test with no multiple-comparison correction.

## Setup

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import binomtest

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MAIN_DIR = '/content/drive/MyDrive/Frugal AI Project/'
OUTPUT_DIR = os.path.join(MAIN_DIR, 'Subjective Track')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS_PATH = os.path.join(MAIN_DIR, 'sampled_dataset.csv')
BASE_DIR    = os.path.join(MAIN_DIR, 'generated_portraits_hashed_final/generated_portraits_hashed')
CAP_DIR     = os.path.join(MAIN_DIR, 'generated_cap')
GLASSES_DIR = os.path.join(MAIN_DIR, 'generated_glasses')
MASK_DIR    = os.path.join(MAIN_DIR, 'generated_masks')

IMAGE_TYPES = ['base', 'cap', 'glasses', 'mask']
ACCESSORIES = ['cap', 'glasses', 'mask']
JOBS        = ['software_engineer', 'nurse', 'construction_worker']

MODEL_ID = 'google/siglip2-giant-opt-patch16-384'
RAW_PATH = os.path.join(OUTPUT_DIR, 'siglip2_raw_scores.jsonl')
PROMPTS_PATH = os.path.join(OUTPUT_DIR, 'siglip2_prompts.json')

In [ ]:
REQUIRED = ['HASH', 'ETHNICITY', 'GENDER', 'AGE']
df = pd.read_csv(LABELS_PATH)
assert set(REQUIRED) <= set(df.columns), f"missing columns, found {list(df.columns)}"
df = df[REQUIRED].copy()
assert df['HASH'].is_unique, "duplicate HASH values"
print("identities:", len(df))

def image_path(hash_, image_type):
    if image_type == 'base':    return f"{BASE_DIR}/{hash_}.jpg"
    if image_type == 'cap':     return f"{CAP_DIR}/{hash_}_hat.png"
    if image_type == 'glasses': return f"{GLASSES_DIR}/{hash_}_glasses.png"
    if image_type == 'mask':    return f"{MASK_DIR}/{hash_}_mask.png"
    raise ValueError(image_type)

missing = [(h, t) for h in df['HASH'] for t in IMAGE_TYPES if not os.path.exists(image_path(h, t))]
assert not missing, f"{len(missing)} image files not found, first ones: {missing[:3]}"
print("all", len(df) * len(IMAGE_TYPES), "image files found")

identities: 720
all 2880 image files found


## Prompts
Each of the 4 templates of Hausladen et al., "A photo of a [adjective] person.", "A [adjective] person.", "This is a [adjective] person.", "Cropped face photo of a [adjective] person.", is filled in twice:
- neutral prompt, without the adjective: "a photo of a person."
- job prompt, with "suitable for a job as a ..." after "person": "a photo of a person suitable for a job as a nurse."

The article before the job is "an" before a vowel and "a" otherwise. All text is lowercased. 4 templates x (1 neutral + 3 jobs) = 16 prompts.

In [ ]:
JOB_TITLE = {
    'software_engineer':   'software engineer',
    'nurse':               'nurse',
    'construction_worker': 'construction worker',
}
assert list(JOB_TITLE) == JOBS

TEMPLATES = [
    "A photo of a person{suffix}.",
    "A person{suffix}.",
    "This is a person{suffix}.",
    "Cropped face photo of a person{suffix}.",
]

def article(word):
    return 'an' if word[0].lower() in 'aeiou' else 'a'

SUFFIX = {'neutral': ''}
for job in JOBS:
    title = JOB_TITLE[job]
    SUFFIX[job] = f" suitable for a job as {article(title)} {title}"

PROMPTS = []
for prompt_set, suffix in SUFFIX.items():
    for ti, template in enumerate(TEMPLATES):
        PROMPTS.append({'prompt_set': prompt_set, 'template_index': ti,
                        'text': template.format(suffix=suffix).lower()})

TEXTS = [p['text'] for p in PROMPTS]
PROMPT_INDEX = {}
for k, p in enumerate(PROMPTS):
    PROMPT_INDEX.setdefault(p['prompt_set'], []).append(k)
assert all(len(v) == len(TEMPLATES) for v in PROMPT_INDEX.values())

with open(PROMPTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(PROMPTS, f, indent=1, ensure_ascii=False)

print(len(TEXTS), "prompts")
for t in TEXTS:
    print(t)

16 prompts
a photo of a person.
a person.
this is a person.
cropped face photo of a person.
a photo of a person suitable for a job as a software engineer.
a person suitable for a job as a software engineer.
this is a person suitable for a job as a software engineer.
cropped face photo of a person suitable for a job as a software engineer.
a photo of a person suitable for a job as a nurse.
a person suitable for a job as a nurse.
this is a person suitable for a job as a nurse.
cropped face photo of a person suitable for a job as a nurse.
a photo of a person suitable for a job as a construction worker.
a person suitable for a job as a construction worker.
this is a person suitable for a job as a construction worker.
cropped face photo of a person suitable for a job as a construction worker.


## Model

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel

assert torch.cuda.is_available(), "no GPU found"
device = 'cuda'
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

def score_image(path):
    image = Image.open(path).convert('RGB')
    inputs = processor(text=TEXTS, images=image, padding="max_length", max_length=64,
                       return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits_per_image[0].float().cpu().tolist()

test = score_image(image_path(df['HASH'].iloc[0], 'base'))
assert len(test) == len(TEXTS), f"expected {len(TEXTS)} logits, got {len(test)}"
print("model loaded, one image scored:", len(test), "logits")

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/101k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

model loaded, one image scored: 16 logits


## Score all images
Every image's raw logits are appended to siglip2_raw_scores.jsonl as soon as they are computed.

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

done = set()
if os.path.exists(RAW_PATH):
    done = {(r['hash'], r['image_type']) for r in load_jsonl(RAW_PATH)}
print("already done:", len(done), "of", len(df) * len(IMAGE_TYPES))

with open(RAW_PATH, 'a', encoding='utf-8') as f:
    for hash_ in tqdm(df['HASH'], desc="identities"):
        for image_type in IMAGE_TYPES:
            if (hash_, image_type) in done:
                continue
            logits = score_image(image_path(hash_, image_type))
            f.write(json.dumps({'hash': hash_, 'image_type': image_type, 'logits': logits}) + '\n')
            f.flush()

already done: 0 of 2880


identities:   0%|          | 0/720 [00:00<?, ?it/s]

## Job scores
For each image and job: job logit = mean logit of the 4 job prompts, neutral logit = mean logit of the 4 neutral prompts, score = job logit minus neutral logit. A higher score means the image matches "a person suitable for a job as a ..." better, relative to "a person". SigLIP 2's logits are the cosine similarity times one scale plus one bias, shared by all prompts (as far as I know), so the bias cancels in the difference and the score is the scaled difference of the average cosine similarities.

In [ ]:
raw = load_jsonl(RAW_PATH)
expected_rows = len(df) * len(IMAGE_TYPES)
assert len(raw) == expected_rows, f"expected {expected_rows} scored images, found {len(raw)}"
assert len({(r['hash'], r['image_type']) for r in raw}) == expected_rows, "duplicate scored images"

rows = []
for r in raw:
    logits = np.asarray(r['logits'], dtype=float)
    assert len(logits) == len(TEXTS)
    neutral_logit = logits[PROMPT_INDEX['neutral']].mean()
    for job in JOBS:
        job_logit = logits[PROMPT_INDEX[job]].mean()
        rows.append({
            'hash': r['hash'], 'job': job, 'image_type': r['image_type'],
            'job_logit': job_logit, 'neutral_logit': neutral_logit,
            'score': job_logit - neutral_logit,
        })

scores = pd.DataFrame(rows)
scores.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_job_scores.csv'), index=False)
print(len(scores), "job scores")
scores.head()

8640 job scores


,hash,job,image_type,job_logit,neutral_logit,score
0,09534c3a20eaad55,software_engineer,base,-0.697704,0.269027,-0.966731
1,09534c3a20eaad55,nurse,base,-2.819224,0.269027,-3.088250
2,09534c3a20eaad55,construction_worker,base,-3.889535,0.269027,-4.158562
3,09534c3a20eaad55,software_engineer,cap,-4.044856,-2.802632,-1.242224
4,09534c3a20eaad55,nurse,cap,-5.527443,-2.802632,-2.724811


## Checks
- Base job score per job (mean, standard deviation, n): the reference for every accessory result.
- Mean neutral logit per image type: how much each accessory, by itself, changes how well the image matches "a person". This is the part that the subtraction removes from the job score.

Unclear and none rates not exist for SigLIP 2, since it always gives a score.

In [ ]:
# [section 6: checks]
base_scores = (scores[scores['image_type'] == 'base']
               .groupby('job')['score'].agg(mean_score='mean', sd_score='std', n='size')
               .reindex(JOBS).round(3))
print("Base job score (logit units):"); print(base_scores.to_string())

neutral_by_type = (scores.drop_duplicates(['hash', 'image_type'])
                   .groupby('image_type')['neutral_logit'].agg(mean_neutral_logit='mean')
                   .reindex(IMAGE_TYPES).round(3))
print("\nMean neutral logit per image type:"); print(neutral_by_type.to_string())

base_scores.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_check_base_scores.csv'))
neutral_by_type.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_check_neutral_logit.csv'))

Base job score (logit units):
                     mean_score  sd_score    n
job                                           
software_engineer        -0.771     1.258  720
nurse                    -3.213     1.015  720
construction_worker      -3.456     1.197  720

Mean neutral logit per image type:
            mean_neutral_logit
image_type                    
base                    -0.215
cap                     -2.003
glasses                 -2.263
mask                    -3.274


## RQ2: exact sign test per job x accessory
For each identity, compare the job score on the base image with the job score on the accessory image.
- down = lower score with the accessory, up = higher score, same = equal score.
- Mean change = mean score with accessory minus mean score on base (logit units).
- p-value: exact two-sided binomial test of down out of (down + up) against 0.5, ties dropped. If down + up = 0 no score changed and the p-value is NaN.

The 9 tests (3 jobs x 3 accessories) are not corrected for multiple comparisons.

In [ ]:
def sign_test(base_score, accessory_score):
    base_score = np.asarray(base_score, dtype=float)
    accessory_score = np.asarray(accessory_score, dtype=float)
    diff = accessory_score - base_score
    down = int((diff < 0).sum())
    up = int((diff > 0).sum())
    p = binomtest(down, down + up, 0.5).pvalue if (down + up) > 0 else float('nan')
    return {
        'n': len(diff),
        'mean_base': round(base_score.mean(), 3),
        'mean_accessory': round(accessory_score.mean(), 3),
        'n_down': down,
        'n_same': len(diff) - down - up,
        'n_up': up,
        'mean_change': round(diff.mean(), 3),
        'p_value': p,
    }

def paired_results(scores, hashes=None):
    rows = []
    for job in JOBS:
        g = scores[scores['job'] == job]
        if hashes is not None:
            g = g[g['hash'].isin(hashes)]
        wide = g.pivot(index='hash', columns='image_type', values='score')
        for accessory in ACCESSORIES:
            rows.append({'job': job, 'accessory': accessory,
                         **sign_test(wide['base'].values, wide[accessory].values)})
    return pd.DataFrame(rows)

overall = paired_results(scores)
overall.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_sign_test_overall.csv'), index=False)
overall

,job,accessory,n,mean_base,mean_accessory,n_down,n_same,n_up,mean_change,p_value
0,software_engineer,cap,720,-0.771,-0.536,275,0,445,0.235,2.499373e-10
1,software_engineer,glasses,720,-0.771,-0.502,255,0,465,0.268,4.347948e-15
2,software_engineer,mask,720,-0.771,-2.928,708,0,12,-2.158,1.362902e-191
3,nurse,cap,720,-3.213,-2.424,87,0,633,0.789,3.425772e-103
4,nurse,glasses,720,-3.213,-3.077,288,0,432,0.136,8.969966e-08
5,nurse,mask,720,-3.213,0.104,0,0,720,3.316,3.626044e-217
6,construction_worker,cap,720,-3.456,-1.552,4,0,716,1.904,4.049049e-207
7,construction_worker,glasses,720,-3.456,-4.122,634,0,86,-0.666,4.691273e-104
8,construction_worker,mask,720,-3.456,-3.761,492,0,228,-0.306,3.588343e-23


## RQ3: the same test split by ethnicity, gender and age
n is the number of identities in the group. The tests are not corrected for multiple comparisons.

In [ ]:
parts = []
for group_var in ['ETHNICITY', 'GENDER', 'AGE']:
    for value in sorted(df[group_var].unique()):
        hashes = set(df.loc[df[group_var] == value, 'HASH'])
        t = paired_results(scores, hashes)
        t.insert(0, 'group_var', group_var)
        t.insert(1, 'group', value)
        parts.append(t)

by_group = pd.concat(parts, ignore_index=True)
by_group.to_csv(os.path.join(OUTPUT_DIR, 'siglip2_sign_test_by_group.csv'), index=False)
by_group

,group_var,group,job,accessory,n,mean_base,mean_accessory,n_down,n_same,n_up,mean_change,p_value
0,ETHNICITY,African,software_engineer,cap,180,-1.284,-0.928,56,0,124,0.355,4.367142e-07
1,ETHNICITY,African,software_engineer,glasses,180,-1.284,-1.026,63,0,117,0.257,6.950541e-05
2,ETHNICITY,African,software_engineer,mask,180,-1.284,-3.360,176,0,4,-2.076,5.646862e-47
3,ETHNICITY,African,nurse,cap,180,-3.221,-2.371,18,0,162,0.850,3.740080e-30
4,ETHNICITY,African,nurse,glasses,180,-3.221,-3.008,59,0,121,0.213,4.434449e-06
...,...,...,...,...,...,...,...,...,...,...,...,...
76,AGE,55,nurse,glasses,240,-3.624,-3.474,91,0,149,0.150,2.192937e-04
77,AGE,55,nurse,mask,240,-3.624,-0.259,0,0,240,3.365,1.131960e-72
78,AGE,55,construction_worker,cap,240,-3.987,-1.882,0,0,240,2.106,1.131960e-72
79,AGE,55,construction_worker,glasses,240,-3.987,-4.440,192,0,48,-0.453,1.386936e-21
